In [1]:
import numpy as np
import matplotlib.pyplot as plt
from tensorly.decomposition import TensorTrain
from tensorly.tt_tensor import tt_to_tensor
import tensorly as tl

# Set backend
tl.set_backend('numpy')

# Define 3D function
def f(xyz, c=5):
    return 1 / (1 + c**2 * np.sum(xyz**2, axis=1))

# Generate Chebyshev polynomials
def chebyshev_polys(x, deg):
    T = np.zeros((deg+1, len(x)))
    T[0] = 1
    if deg > 0:
        T[1] = x
    for k in range(2, deg+1):
        T[k] = 2 * x * T[k-1] - T[k-2]
    return T

# Generate coefficient tensor (N+1 should be TT-factorable)
def generate_coeff_tensor(N, c):
    k = np.arange(N+1)
    nodes = np.cos((2*k + 1) * np.pi / (2*(N+1)))
    X, Y, Z = np.meshgrid(nodes, nodes, nodes, indexing="ij")
    coords = np.stack([X.ravel(), Y.ravel(), Z.ravel()], axis=1)
    F = f(coords, c).reshape((N+1, N+1, N+1))

    Tx = chebyshev_polys(nodes, N)
    A = np.kron(Tx.T, np.kron(Tx.T, Tx.T))
    F_flat = F.ravel()
    c_flat, *_ = np.linalg.lstsq(A, F_flat, rcond=None)
    C = c_flat.reshape((N+1, N+1, N+1))
    return C, nodes

# Evaluate interpolated function using Chebyshev coefficients
def evaluate_interp(C, x_nodes, N, resolution=50):
    xx = np.linspace(-1, 1, resolution)
    yy = np.linspace(-1, 1, resolution)
    zz = np.linspace(-1, 1, resolution)
    Tx = chebyshev_polys(xx, N)
    Ty = chebyshev_polys(yy, N)
    Tz = chebyshev_polys(zz, N)
    F = np.zeros((resolution, resolution, resolution))
    for i in range(N+1):
        for j in range(N+1):
            for k in range(N+1):
                F += C[i, j, k] * np.einsum("i,j,k->ijk", Tx[i], Ty[j], Tz[k])
    return F

# Exact ground truth
def compute_exact_function_grid(f, c, resolution=50):
    xx = np.linspace(-1, 1, resolution)
    yy = np.linspace(-1, 1, resolution)
    zz = np.linspace(-1, 1, resolution)
    X, Y, Z = np.meshgrid(xx, yy, zz, indexing="ij")
    coords = np.stack([X.ravel(), Y.ravel(), Z.ravel()], axis=1)
    return f(coords, c).reshape((resolution, resolution, resolution))

# Configs
N = 63  # N+1 = 64, which is 2^6 and easily factorizable
c = 5
resolution = 10

# Generate Chebyshev coefficient tensor
C, x_nodes = generate_coeff_tensor(N, c)

# Tensorize C to shape (4,4,4,4,4,4) or any 6-factor shape such that product = 64^3
C_tensorized = C.reshape((8, 8, 8, 8, 8, 8))

# Decompose using Tensor Train
tt_model = TensorTrain(rank='same', verbose=False)
tt_cores = tt_model.fit_transform(C_tensorized)

# # Reconstruct from TT cores
# C_reconstructed_tensor = tt_to_tensor(tt_cores)
# C_reconstructed = C_reconstructed_tensor.reshape((N+1, N+1, N+1))

# # Evaluate interpolated function using reconstructed coefficients
# F_tt = evaluate_interp(C_reconstructed, x_nodes, N, resolution)

# # Compute ground truth
# F_true = compute_exact_function_grid(f, c, resolution)

# # Compute errors
# diff = F_true - F_tt
# rmse = np.sqrt(np.mean(diff**2))
# maxe = np.max(np.abs(diff))

# print(f"TT Decomposition → RMSE: {rmse:.2e}, MaxE: {maxe:.2e}")

# # Optional: show mid-slice comparison
# mid = resolution // 2
# fig, axs = plt.subplots(1, 2, figsize=(12, 5))
# axs[0].imshow(F_true[mid], origin='lower', cmap='plasma')
# axs[0].set_title('True Function (Z-mid slice)')
# axs[1].imshow(F_tt[mid], origin='lower', cmap='plasma')
# axs[1].set_title('TT Interpolation (Z-mid slice)')
# plt.tight_layout()
# plt.show()


: 